In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount = True)
dir_path = '/content/drive/MyDrive/02740_data/recitation_2' # Path to the folder

os.chdir(dir_path)

# Convolution

- In the context of signal processing, image processing, and neural networks, convolution involves applying a kernel (or filter) to an input to extract features or transform data.

## Parameters of Convolution Operations

- **kernel size**: the dimensions kernel applied to the input.
- **padding size**: padding is the process of adding extra pixels (usually zeros) around the border of the input.
- **stride size**: Stride refers to how much the filter moves during convolution. For a stride of 1, the filter moves one pixel at a time; for a stride of 2, it moves two pixels. Larger strides reduce the spatial size of the output feature map.


The spatial size of the output (along one dimension) is:

$$
n_{\mathrm{out}} = \left\lfloor \frac{n_{\mathrm{in}} + 2p - k}{s} \right\rfloor + 1
$$

where:

- $n_{\mathrm{in}}$: number of input features
- $n_{\mathrm{out}}$: number of output features
- $k$: convolution kernel size
- $p$: convolution padding size
- $s$: convolution stride size


## Perform 2D Convolution Using Different Modules

In [ ]:
import cv2
import numpy as np
import scipy
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Load the image
img = cv2.imread('endoplasmic_reticulum.jpg')
plt.title('Original Image')
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')


# Convert image to grayscale for simplicity
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print(img_gray.shape)

In [ ]:
# Define a sample kernel (e.g., a simple edge detection kernel)
# This is a Laplacian kernel (a type of second-order derivative operator)
kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
])

# cv2: OpenCV uses filter2D and performs convolution with the kernel flipped by default
output_cv2 = cv2.filter2D(img_gray, ddepth = -1, kernel = kernel)
print(output_cv2.shape)

# SciPy: Use convolve2d with 'valid' mode
output_scipy = scipy.signal.convolve2d(img_gray, kernel, mode='valid')
print(output_scipy.shape)

# Pytorch:
# Step 1: Convert input and kernel to PyTorch tensors and reshape
input_tensor = torch.tensor(img_gray, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions
kernel_tensor = torch.tensor(kernel, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# Step 2: Perform convolution
output_pytorch = F.conv2d(input_tensor, kernel_tensor)
output_pytorch = torch.Tensor.numpy(output_pytorch.squeeze(0).squeeze(0))
print(output_pytorch.shape)

output_pytorch = F.conv2d(input_tensor, kernel_tensor, padding=1)
output_pytorch = torch.Tensor.numpy(output_pytorch.squeeze(0).squeeze(0))
print(output_pytorch.shape)


In [ ]:
# Apply convolution using cv2.filter2D
convoluted_img = cv2.filter2D(src=img_gray, ddepth=-1, kernel=kernel)

# Show the images
plt.figure(figsize=(10, 5))

# Original Image
plt.subplot(1, 2, 1)
plt.title('Original Image')
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')

# Convoluted Image
plt.subplot(1, 2, 2)
plt.title('Convoluted Image')
plt.imshow(convoluted_img, cmap='gray')
plt.axis('off')

plt.show()

# Save the convoluted image
cv2.imwrite('convoluted_image.jpg', convoluted_img)


### Gaussian Kernel

A Gaussian kernel is used in image processing for tasks such as blurring, noise reduction, and edge detection. OpenCV provides a convenient function to create and apply Gaussian kernels.

Usage in Bioimages: 1) Noise Reduction and Smoothing

2) Preprocessing for Feature Detection and Morphological Operations

Below is a sample code demonstrating the use of OpenCV's Gaussian kernel implementation:

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_gaussian = cv2.resize(img_gray, (100, 100))

# Define a function to compute and display the Gaussian blur and kernel values
def apply_gaussian_blur(input_array, ksize, sigma):
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(input_array, (ksize, ksize), sigma)

    # Manually calculate the Gaussian kernel (using OpenCV's getGaussianKernel function)
    gaussian_kernel = cv2.getGaussianKernel(ksize, sigma)  # 1D Gaussian kernel
    gaussian_kernel_2d = np.outer(gaussian_kernel, gaussian_kernel)  # Convert to 2D

    # Display the results
    print(f"Gaussian Kernel with sigma={sigma}:\n", gaussian_kernel_2d)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.title(f"Original Image")
    plt.imshow(input_array, cmap='gray')
    plt.colorbar()

    plt.subplot(1, 2, 2)
    plt.title(f"Blurred with sigma={sigma}")
    plt.imshow(blurred, cmap='gray')
    plt.colorbar()

    plt.show()

# Apply Gaussian blur with different values of sigma
apply_gaussian_blur(img_gaussian, ksize=5, sigma=0.5)
apply_gaussian_blur(img_gaussian, ksize=5, sigma=1)
apply_gaussian_blur(img_gaussian, ksize=5, sigma=2)

## Edge Detection


Edge detection is a fundamental technique in image processing for identifying boundaries within images. Using the `filter` module from Scikit-Image (SKImage), we can apply various edge detection algorithms, such as the Sobel or Canny filters. These filters highlight significant edges in the image, which are crucial for tasks like object detection and image segmentation.

In addition to using predefined filters, we can also implement edge detection by calculating the image gradient. Below is a sample code demonstrating how to compute first and second derivatives for edge detection:


## Edge Detection Operators

Edge detection operators are used to identify significant changes in intensity in an image, which typically correspond to edges. Scikit-Image (SKImage) provides various filters for edge detection, such as Sobel, Roberts, and Prewitt. These filters highlight edges by computing the gradient of the image intensity at each pixel.

Below is a sample code demonstrating the use of SKImage filters for edge detection:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage.data import camera
from skimage.filters import roberts, sobel, sobel_h, sobel_v, prewitt, prewitt_v, prewitt_h

image_ori = cv2.imread('zebrafish_embryos.png')
image_ori = cv2.cvtColor(image_ori, cv2.COLOR_BGR2GRAY)

image = cv2.GaussianBlur(image_ori, (3,3), 0)

edge_roberts = roberts(image)
edge_sobel = sobel(image)

fig, ax = plt.subplots(ncols=3, sharex=True, sharey=True, figsize=(8, 4))

ax[0].imshow(image_ori, cmap = plt.cm.gray)
ax[0].set_title('Origial')


ax[1].imshow(edge_roberts, cmap=plt.cm.gray)
ax[1].set_title('Roberts Edge Detection')

ax[2].imshow(edge_sobel, cmap=plt.cm.gray)
ax[2].set_title('Sobel Edge Detection')

for a in ax:
    a.axis('off')

plt.tight_layout()
plt.show()

#### Edge Detection using self-defined kernels

OpenCV provides numerous edge detection operators to highlight edges in an image. Some common edge detection techniques include Canny, Sobel, and Laplacian.

Below is a sample code demonstrating the use of OpenCV for edge detection using these filters:

In [ ]:
# Roberts kernels
roberts_x = np.array([[1, 0],
                      [0, -1]], dtype=np.float32)

roberts_y = np.array([[0, 1],
                      [-1, 0]], dtype=np.float32)

# Sobel kernels
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)

sobel_y = np.array([[-1, -2, -1],
                    [0,  0,  0],
                    [1,  2,  1]], dtype=np.float32)

# Prewitt kernels
prewitt_x = np.array([[-1, 0, 1],
                      [-1, 0, 1],
                      [-1, 0, 1]], dtype=np.float32)

prewitt_y = np.array([[-1, -1, -1],
                      [0,   0,  0],
                      [1,   1,  1]], dtype=np.float32)

# Function to apply kernel
def apply_kernel(image, kernel, name):
    output = cv2.filter2D(image, -1, kernel)
    plt.imshow(output, cmap='gray')
    plt.title(f"{name} Edge Detection")
    plt.show()

# Apply Roberts kernels
apply_kernel(image, roberts_x, "Roberts X")
apply_kernel(image, roberts_y, "Roberts Y")

# Apply Sobel kernels
apply_kernel(image, sobel_x, "Sobel X")
apply_kernel(image, sobel_y, "Sobel Y")

# Apply Prewitt kernels
apply_kernel(image, prewitt_x, "Prewitt X")
apply_kernel(image, prewitt_y, "Prewitt Y")

In [ ]:
# Combine to get the magnitude
image_x = cv2.filter2D(image, cv2.CV_32F, sobel_x)
image_y = cv2.filter2D(image, cv2.CV_32F, sobel_y)

# Compute gradient magnitude
gradient_magnitude = np.sqrt(image_x**2 + image_y**2)

# Compute gradient direction (in radians)
gradient_direction = np.arctan2(image_y, image_x)

plt.imshow(gradient_magnitude, cmap='gray')
plt.show()
plt.imshow(gradient_direction, cmap='gray')
plt.colorbar()
plt.show()

## Canny edge detection algorithm
It was developed by John F. Canny in 1986. Canny operator is a multi-stage algorithm that detects wide range of edges.

#### **Stages**

1. Smoothing for noise removal.
2. Finding Gradients.
3. None-maximum suppression.
4. Double Thresholding.
5. Edge Tracking by hysteresis threshold.

In [ ]:
# Edge enhancement by none-maximum suppression

def non_maximum_suppression(gradient_magnitude, gradient_direction):
    """
    Performs non-maximum suppression on the gradient magnitude image.

    Args:
    - gradient_magnitude: The gradient magnitude image (2D numpy array).
    - gradient_direction: The gradient direction image (2D numpy array in radians).

    Returns:
    - thin_edges: Output edge map after non-maximum suppression.
    """
    # Get the shape of the image
    rows, cols = gradient_magnitude.shape

    # Create an array to hold the thinned edges (initialized with zeros)
    thin_edges = np.zeros((rows, cols), dtype=np.float32)

    # Convert gradient directions from radians to degrees and normalize to [0, 180]
    gradient_direction = gradient_direction * 180.0 / np.pi
    gradient_direction[gradient_direction < 0] += 180

    # Iterate over every pixel (ignoring the border pixels)
    for i in range(1, rows - 1):
        for j in range(1, cols - 1):
            # Determine the direction of the current pixel (rounded to one of the four possible directions)
            angle = gradient_direction[i, j]
            current_pixel = gradient_magnitude[i, j]

            # Identify neighboring pixels in the direction of the gradient
            if (0 <= angle < 22.5) or (157.5 <= angle <= 180):
                neighbor1 = gradient_magnitude[i, j - 1]  # Left neighbor
                neighbor2 = gradient_magnitude[i, j + 1]  # Right neighbor
            elif 22.5 <= angle < 67.5:
                neighbor1 = gradient_magnitude[i - 1, j + 1]  # Top-right neighbor
                neighbor2 = gradient_magnitude[i + 1, j - 1]  # Bottom-left neighbor
            elif 67.5 <= angle < 112.5:
                neighbor1 = gradient_magnitude[i - 1, j]  # Top neighbor
                neighbor2 = gradient_magnitude[i + 1, j]  # Bottom neighbor
            elif 112.5 <= angle < 157.5:
                neighbor1 = gradient_magnitude[i - 1, j - 1]  # Top-left neighbor
                neighbor2 = gradient_magnitude[i + 1, j + 1]  # Bottom-right neighbor

            # Keep the pixel only if it is greater than both neighbors
            if current_pixel >= neighbor1 and current_pixel >= neighbor2:
                thin_edges[i, j] = current_pixel
            else:
                thin_edges[i, j] = 0  # Suppress the pixel

    return thin_edges

edges = non_maximum_suppression(gradient_magnitude, gradient_direction)
plt.imshow(edges, cmap='gray')
plt.show()

In [ ]:
# Hysteresis Thresholding: to link detected edge points while minimizing breakage

def hysteresis_thresholding(gradient_magnitude, weak, strong, low_threshold, high_threshold):
    """
    Apply Hysteresis Thresholding to an edge map.

    Args:
    - gradient_magnitude: The gradient magnitude image (2D numpy array).
    - weak: Value representing weak edges.
    - strong: Value representing strong edges.
    - low_threshold: Lower bound for weak edges.
    - high_threshold: Upper bound for strong edges.

    Returns:
    - final_edges: Edge map after hysteresis thresholding.
    """
    # Get the dimensions of the image
    rows, cols = gradient_magnitude.shape

    # Initialize the edge map
    final_edges = np.zeros((rows, cols), dtype=np.float32)

    # Identify strong and weak edges
    strong_row, strong_col = np.where(gradient_magnitude >= high_threshold)
    weak_row, weak_col = np.where((gradient_magnitude >= low_threshold) & (gradient_magnitude < high_threshold))

    # Set strong edges
    final_edges[strong_row, strong_col] = strong

    # Define 8-connectivity for neighboring pixels
    dx = [-1, -1, -1, 0, 0, 1, 1, 1]
    dy = [-1, 0, 1, -1, 1, -1, 0, 1]

    # Iterate over all weak edge pixels
    for i, j in zip(weak_row, weak_col):
        # Check if any of the neighbors are strong edges
        for k in range(8):
            ni, nj = i + dx[k], j + dy[k]
            if 0 <= ni < rows and 0 <= nj < cols:  # Ensure within bounds
                if final_edges[ni, nj] == strong:
                    final_edges[i, j] = strong
                    break  # If connected to a strong edge, mark as strong

    return final_edges

# Set thresholds for hysteresis
low_threshold = 10  # Can be tuned based on the image
high_threshold = 50  # Can be tuned based on the image

# Define weak and strong edge values
weak = 100
strong = 200

# Apply hysteresis thresholding
final_edges = hysteresis_thresholding(edges, weak, strong, low_threshold, high_threshold)

# Display the final edge map
plt.imshow(final_edges, cmap='gray')
plt.show()

In [ ]:
# Edge detection kernel for arbitary size

import numpy as np
import cv2

sobel5x = cv2.getDerivKernels(1, 0, 5)
print(np.outer(sobel5x[0], sobel5x[1]))

sobel5y = cv2.getDerivKernels(0, 1, 5)
print(np.outer(sobel5y[0], sobel5y[1]))

sobel7x = cv2.getDerivKernels(1, 0, 7)
print(np.outer(sobel7x[0], sobel7x[1]))

sobel7y = cv2.getDerivKernels(0, 1, 7)
print(np.outer(sobel7y[0], sobel7y[1]))

- Small kernel (3x3): You will detect more details and possibly more noise. The edges will be sharper, but finer structures and noise may also appear as edges.
- Large kernel (5x5, 7x7): The larger kernel results in smoother edges, better noise suppression, and reduced sensitivity to small-scale features. Large-scale edges are more prominent, but finer details are less noticeable.

In [ ]:
# Examples for different kernel size

sobelx_3 = cv2.Sobel(src=image, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=3) # X Sobel Edge Detection of kernel size 3
sobelx_5 = cv2.Sobel(src=image, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=5) # X Sobel Edge Detection of kernel size 5
sobelx_7 = cv2.Sobel(src=image, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=7) # X Sobel Edge Detection of kernel size 7

# Display Sobel Edge Detection Images

plt.imshow(np.abs(sobelx_3), cmap='gray')
plt.colorbar()
plt.show()
plt.imshow(np.abs(sobelx_5), cmap='gray')
plt.colorbar()
plt.show()
plt.imshow(np.abs(sobelx_7), cmap='gray')
plt.colorbar()
plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

image = cv2.imread('zebrafish_embryos.png', cv2.IMREAD_GRAYSCALE)

edges_canny = cv2.Canny(image, 100, 200)
sobelx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=5)
sobely = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=5)
sobel_combined = cv2.magnitude(sobelx, sobely)

plt.figure(figsize=(12, 8))

plt.subplot(2, 3, 1)
plt.imshow(edges_canny, cmap='gray')
plt.title('Canny Edge Detection')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.imshow(sobel_combined, cmap='gray')
plt.title('Sobel Edge Detection')
plt.axis('off')

plt.tight_layout()
plt.show()



## Thresholding

Thresholding is a technique in image processing used to create binary images from grayscale images. By setting pixel values above a certain threshold to one value and those below to another, we can highlight specific features in an image. Hysteresis thresholding is a two-threshold approach that helps in edge detection by connecting weak edges.

Below is a sample code demonstrating thresholding using the Sobel edge detection and applying hysteresis thresholding:

In [ ]:
import matplotlib.pyplot as plt
from skimage import data, filters

low = 50
high = 200

lowt = (image > low).astype(int)
hight = (image > high).astype(int)

hyst = filters.apply_hysteresis_threshold(image, low, high)

hyst_img = hight + hyst

fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True)
ax = axes.ravel()

ax[0].imshow(image, cmap=plt.cm.gray)
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(lowt, cmap=plt.cm.gray)
ax[1].set_title(f'Low Threshold > {low}')
ax[1].axis('off')

ax[2].imshow(hight, cmap=plt.cm.gray)
ax[2].set_title(f'High Threshold < {high}')
ax[2].axis('off')

ax[3].imshow(hyst, cmap=plt.cm.gray)
ax[3].set_title('Hysteresis Thresholding')
ax[3].axis('off')

ax[4].imshow(hyst_img, cmap=plt.cm.gray)
ax[4].set_title('Hysteresis + High Threshold')
ax[4].axis('off')

fig.delaxes(ax[5])

plt.tight_layout()
plt.show()


#### Image Thresholding using OpenCV

OpenCV provides several types of thresholding, including binary, binary inverse, truncation, to zero, and to zero inverse.

Below is a sample code demonstrating the use of various OpenCV thresholding techniques:




#### Adaptive Thresholding

In adaptive thresholding, the threshold value is calculated for smaller regions of the image. This means that different parts of the image can have different threshold values.
This method is useful for images with varying lighting conditions. OpenCV provides functions for adaptive mean thresholding and adaptive Gaussian thresholding.

Below is a sample code demonstrating the use of adaptive thresholding with OpenCV:

In [ ]:
adaptive_mean = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 2)
adaptive_gaussian = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

plt.figure(figsize=(10, 4))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(adaptive_mean, cmap='gray')
plt.title('Adaptive Mean Thresholding')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(adaptive_gaussian, cmap='gray')
plt.title('Adaptive Gaussian Thresholding')
plt.axis('off')

plt.tight_layout()
plt.show()

#### OTSU'S Thresholding

Otsu's thresholding is a global thresholding technique that automatically determines the optimal threshold value for an image. It does so by minimizing the within-class variance (the variance within the foreground and background pixels). This method is particularly effective for bimodal images, where the histogram has two distinct peaks representing the foreground and background.

Below is a sample code demonstrating the use of Otsu's thresholding with OpenCV:

In [ ]:
_, otsu_thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(otsu_thresh, cmap='gray')
plt.title("Otsu's Binarization")
plt.axis('off')

plt.tight_layout()
plt.show()